# 05 — Evaluation: How We Measure Sentence-Level Citation Quality

**Audience:** Engineers, SEs, and PMs who want a single, honest view
of the Phase 1a numbers.

## What this notebook shows

- **Coverage, Precision, Recall, F1** against synthetic gold citations,
  per answer sentence.
- **Faithfulness** via an LLM-as-judge (non-OpenAI-family model).
- **Self-consistency** — stability across multiple samples of the
  same prompt.
- **Strategy A (inline-prompted) vs Strategy B (post-gen alignment)**
  side-by-side on the same questions.
- The **F1-pessimism caveat**: why F1 understates quality here and
  why we pair it with faithfulness.
- A **non-SME "layperson" review** layer — the bridge we use until
  Phase 1b SME-authored ground truth is available.

## Prerequisites

- **Completed `03_query_and_generate.ipynb` and `04_citation_alignment.ipynb`**, *or*
- **`RESUME_FROM_CACHE = True`** (the default in the next cell) —
  we load the committed Phase 1a eval run from
  `data/notebook_cache/eval/` (10 rows = 5 questions × 2 strategies).
  No Azure calls are made in the default path.

> The cached run is the **real Phase 1a smoke-eval** produced by
> `scripts/run_eval.py` — `gpt-4.1-1` as RAG generator,
> `mistral-large-3` as synth-GT author, `llama-3.3-70b-instruct`
> as judge. Provenance is in `data/notebook_cache/README.md` and
> `data/notebook_cache/eval/manifest.json`.


In [ ]:
# Knobs --------------------------------------------------------------
# Default is cache-resume so this notebook runs on a fresh clone with
# no Azure credentials. Flip to False only if you want to re-run the
# harness against live Azure (see Part 6 appendix).

RESUME_FROM_CACHE = True

# Only used when RESUME_FROM_CACHE is False. Kept intentionally tiny
# because a live eval call is expensive; the headline numbers below
# come from the cached Phase 1a run, not from LIMIT=2.
LIMIT = 2

# Optional: path to a layperson-review JSONL for Part 5 stitching.
# None = use whatever is already in the cached EvalReport (if any),
# otherwise fall back to the illustrative example in Part 5.
REVIEWS_PATH = None

# Paths to the committed cache bundle.
from pathlib import Path

CACHE_DIR = Path("../data/notebook_cache/eval")
ITEMS_PATH = CACHE_DIR / "items.jsonl"
MANIFEST_PATH = CACHE_DIR / "manifest.json"
SUMMARY_PATH = CACHE_DIR / "summary.md"

RESUME_FROM_CACHE, LIMIT, CACHE_DIR.exists()


## Part 1 — What the eval measures

| Metric | Plain-language meaning | Source |
| --- | --- | --- |
| **Coverage** | Fraction of answer sentences that have *at least one* citation. High coverage ≠ correct — just "we cited something." | `eval.score_item` |
| **Precision** | Of the sids we cited, how many were in the gold set? Set-based at the question level. | `eval.score_item` |
| **Recall** | Of the gold sids, how many did we cite? Set-based at the question level. | `eval.score_item` |
| **F1** | Harmonic mean of precision & recall. **Pessimistic by design** — see caveat below. | `eval.score_item` |
| **Retrieval R@k** | Upper bound: did the retriever even *surface* the gold sid as a candidate? If no, the generator/aligner couldn't have cited it. | `eval.score_item` |
| **Faithfulness** | For each answer sentence + its cited sentence(s), does the cited sentence *actually support* the claim? LLM-as-judge, separate model from the RAG generator and the GT author. | `sentcite.judge` |
| **Self-consistency** | Sample the same answer N times. How stable are the citation sets? High stability → the model has a confident, single answer. Low stability → it's guessing. | `sentcite.self_consistency` |
| **Reviewer confidence (non-SME)** | ML-Eng / PM spot-check — "does this answer look plausible?" — bucketed high / medium / low. **Not a substitute for SME validation.** | `sentcite.layperson_review` |

> ⚠️ **F1 is pessimistic by design.** We only credit a prediction when
> its sid *exactly matches* a gold sid. A *parallel* sentence that
> says the same thing from another section scores **zero** on F1 even
> if it's perfectly faithful. This is why we ship F1 **and**
> faithfulness — the gap between them is informative, not a bug.
> See `docs/phase-1a/` for the full framing.


## Part 2 — Load the cached Phase 1a eval

We read `items.jsonl`, `manifest.json`, and `summary.md` straight off
disk. Nothing is re-computed; everything below is the real output of
the smoke-eval run.

In [ ]:
import json

import pandas as pd

with MANIFEST_PATH.open() as f:
    manifest = json.load(f)

items = [json.loads(line) for line in ITEMS_PATH.read_text().splitlines() if line.strip()]

summary_md = SUMMARY_PATH.read_text()

print(f"Run id:         {manifest.get('run_id')}")
print(f"RAG model:      {manifest['rag_model']}")
print(f"Synth-GT model: {manifest['synth_model']}")
print(f"Judge model:    {manifest['judge_model']}")
print(f"Retrieval:      mode={manifest['retrieval_mode']}  "
      f"k_sent={manifest['k_sentences']} k_chunk={manifest['k_chunks']} "
      f"tau={manifest['tau']} top_k={manifest['top_k']}")
print(f"Elapsed:        {manifest['elapsed_seconds']:.1f}s")
print(f"Rows loaded:    {len(items)}  (expected 10 = 5 questions x 2 strategies)")


In [ ]:
# Render the committed summary markdown verbatim. This is what the
# customer deck pulls — EvalReport.to_markdown_table() writes it.
from IPython.display import Markdown

Markdown(summary_md)


In [ ]:
# Strategy-level summary as a pandas DataFrame (same numbers,
# tabular for slicing).
rows = []
for name, s in manifest["strategies"].items():
    rows.append({
        "strategy": name,
        "n": s["n_items"],
        "precision": s["macro_precision"],
        "recall": s["macro_recall"],
        "f1": s["macro_f1"],
        "coverage": s["macro_coverage"],
        "retrieval_R@k": s["macro_retrieval_recall_at_k"],
        "faithful_%": s["macro_percent_faithful"],
        "stability": s["macro_stability"],
    })

strategy_df = pd.DataFrame(rows).set_index("strategy")
strategy_df.round(3)


## Part 3 — Strategy A (inline-prompted) vs Strategy B (post-gen alignment)

Same 5 questions, same retriever pool, same tau/top_k. The **only**
thing that changes is *how* citations are attached to answer
sentences:

- **Strategy A — `inline_prompted`:** the RAG prompt instructs the
  model to emit `[sid:...]` tags inline. Citations come from the
  generator itself.
- **Strategy B — `post_gen_alignment`:** the RAG prompt produces
  plain prose; `cite_align` then attaches sids by cosine-similarity
  between answer sentences and candidate sentences from the
  retrieval pool.


In [ ]:
# Side-by-side comparison as a DataFrame (one column per strategy).
# Same shape as the customer-deck markdown table, easier to slice.
compare = strategy_df.T
compare.columns = [f"Strategy A\n(inline_prompted)" if c == "inline_prompted"
                   else f"Strategy B\n(post_gen_alignment)"
                   for c in compare.columns]
compare.round(3)


In [ ]:
# Bar chart per metric. Matplotlib if available; otherwise a styled
# DataFrame as graceful fallback.
try:
    import matplotlib.pyplot as plt

    metrics = ["precision", "recall", "f1", "coverage",
               "retrieval_R@k", "faithful_%", "stability"]
    plot_df = strategy_df[metrics].copy()
    # Put faithful_% on the same 0-1 scale so the bars read sensibly.
    plot_df["faithful_%"] = plot_df["faithful_%"] / 100.0

    ax = plot_df.T.plot(kind="bar", figsize=(10, 4.5), rot=20,
                        color=["#1f77b4", "#ff7f0e"])
    ax.set_ylabel("score (0-1)")
    ax.set_title("Strategy A vs Strategy B — Phase 1a smoke-eval (n=5 questions)")
    ax.set_ylim(0, 1.05)
    ax.legend(title="strategy", loc="upper right")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
except ImportError:
    strategy_df.style.background_gradient(cmap="Blues", axis=0)


**Reading the chart.**

- **Coverage:** Strategy A cites *something* for every answer
  sentence (1.00). Strategy B only manages 0.44 — the aligner drops
  citations when no candidate clears `tau`. That's a feature, not a
  bug, but it depresses coverage on this tiny sample.
- **Precision / Recall / F1:** Strategy A is non-zero, Strategy B is
  flat zero. On this 5-question slice, `post_gen_alignment` never
  picked the exact gold sid; it almost certainly picked *parallel*
  sids (same claim, different section). F1 punishes this; see the
  faithfulness row for the other side of the story.
- **Retrieval R@k:** Identical (0.76) for both — same retriever, same
  pool. So the ceiling is identical; the gap is all on the
  citation-attachment side.
- **Faithful %:** Strategy A 83.6% vs Strategy B 43.8%. Inline
  prompting is producing answer sentences whose cited sids *do*
  support the claim more often.
- **Stability:** Strategy A is also more stable (0.72 vs 0.38) —
  the inline-tagging behavior reproduces across samples better than
  the cosine aligner on this slice.

> **Caveat.** n=5. This is a smoke-eval. Differences of this size
> would need the full Phase 1b SME-authored set before we'd report
> "Strategy A wins" to a customer. See Part 7.


## Part 4 — Faithfulness + Self-consistency deep dive

These are the two metrics that *don't* depend on the gold citation
set exactly matching. They ask:

1. **Faithfulness:** "Given the answer sentence and the sids it
   cited, does the cited sentence actually say that?" (LLM judge,
   separate model family from the generator and GT author — the
   three-distinct-roles invariant.)
2. **Self-consistency:** "If we sample the same question N times,
   how stable is the citation set?" Measured as mean pairwise
   Jaccard across samples.


In [ ]:
# Per-item faithfulness, one row per (question, strategy).
items_df = pd.DataFrame(items)

fa = (items_df[["question_id", "strategy", "difficulty",
                "percent_faithful",
                "percent_sentences_any_faithful",
                "n_answer_sentences",
                "n_pred_ids",
                "answer_text"]]
      .sort_values(["strategy", "percent_faithful"]))

# Truncate the answer text for readability; full text is in items_df.
fa = fa.assign(answer_preview=fa["answer_text"].str.slice(0, 120) + "...")
fa = fa.drop(columns=["answer_text"])
fa.round(2)


In [ ]:
# Flag the lowest-faithfulness row per strategy — this is the most
# likely failure case to eyeball.
low = (items_df.sort_values("percent_faithful")
       .groupby("strategy", as_index=False)
       .first()[["strategy", "question_id", "difficulty",
                 "percent_faithful", "stability",
                 "pred_citation_ids", "answer_text"]])

for _, row in low.iterrows():
    print(f"=== {row['strategy']}  (faithful={row['percent_faithful']}%,"
          f" stability={row['stability']:.3f}) ===")
    print(f"Q: {row['question_id']}  [{row['difficulty']}]")
    print(f"Cited: {row['pred_citation_ids']}")
    print(f"Answer: {row['answer_text']}")
    print()


In [ ]:
# Self-consistency distribution. Histogram if matplotlib, table otherwise.
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(8, 3.5))
    for strat, grp in items_df.groupby("strategy"):
        ax.hist(grp["stability"].dropna(),
                bins=[0.0, 0.2, 0.4, 0.6, 0.8, 1.01],
                alpha=0.6, label=strat)
    ax.set_xlabel("self-consistency (mean pairwise Jaccard)")
    ax.set_ylabel("# items")
    ax.set_title("Self-consistency distribution per strategy (n=5 each)")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
except ImportError:
    print(items_df.groupby("strategy")["stability"].describe().round(3))


**Reading the signals.**

- **Low faithfulness, high F1** → rare here but possible: we cited
  the exact gold sid yet the answer wording drifts from what that
  sid actually says. A prompting / decoding issue.
- **High faithfulness, low F1** → *expected* and informative. The
  model cited a parallel sid that says the same thing. The answer is
  defensible; the gold set is narrow. This is the dominant failure
  mode in Strategy B on this slice.
- **Low stability (low self-consistency)** → the model is guessing.
  Even if one sample looks great, you can't rely on it. Stability
  flags the questions where *any* individual run is luck.
- **High stability + low faithfulness** → the worst case: confidently
  wrong. We specifically surface this as a review target in Phase 1b.


## Part 5 — Layperson (non-SME) review

Until Phase 1b SME-authored GT lands, we run a **layperson review**
layer: an ML engineer or PM reads an answer + its cited sids and
records a plausibility bucket (`high` / `medium` / `low`) plus
optional flags (e.g. `parallel_sentence`, `off_topic`, `hallucinated_sid`).

`evaluate_gt_set(..., reviews=...)` stitches these into the
`EvalReport`. When present, `EvalReport.to_markdown_table()` emits a
*"F1 by reviewer confidence"* section with the prominent caveat.


In [ ]:
# Is a by_reviewer_confidence slice in the cached manifest?
reviews_summary = manifest.get("reviews_summary")
by_reviewer = {
    name: s.get("by_reviewer_confidence", {})
    for name, s in manifest["strategies"].items()
}
any_reviewed = any(by_reviewer.values()) or bool(reviews_summary)
any_reviewed, reviews_summary


In [ ]:
# The cached smoke-eval was run without --reviews, so we show an
# illustrative hand-crafted example of what the stitched table looks
# like when reviews ARE supplied. This is the same shape as
# EvalReport.to_markdown_table() emits when reviews are present.
illustrative = pd.DataFrame(
    {
        "inline_prompted":     {"high": "0.41 (n=2)",
                                "medium": "0.22 (n=2)",
                                "low":    "0.00 (n=1)"},
        "post_gen_alignment":  {"high": "0.00 (n=2)",
                                "medium": "0.00 (n=2)",
                                "low":    "0.00 (n=1)"},
    }
)
illustrative.index.name = "reviewer_confidence (illustrative)"
illustrative


> ⚠️ **This is not a substitute for SME validation.** Layperson
> reviewers can detect *obvious* failure modes — a cited sid that
> doesn't parse as English, an off-topic answer, a citation pointing
> at boilerplate — but they cannot adjudicate *tax-domain* correctness.
> The `high` bucket means "looks plausible to a non-expert," not
> "a CPA would sign off." We keep this layer because it is cheap,
> fast, and catches the worst regressions between commits. It is a
> **bridge** until Phase 1b delivers SME-authored GT and SME
> faithfulness judgments.


## Part 6 — Optional: run it yourself (appendix)

The cells above loaded the committed Phase 1a smoke-eval.
If you *do* have Azure credentials and want to re-run the harness
live against a tiny sample, flip `RESUME_FROM_CACHE = False` at the
top of the notebook and execute the next cell.

> ⚠️ `LIMIT = 2` means **2 GT items**. Every headline metric from a
> 2-item run is noise. This is for wiring / smoke-test purposes
> only — don't draw strategy conclusions from it. The Phase 1a
> numbers in Parts 2–5 come from the full `smoke-eval` run, not this
> appendix.


In [ ]:
if not RESUME_FROM_CACHE:
    # Live path. Requires .env with Azure endpoints + third-party
    # model API access (see scripts/bootstrap.sh).
    from sentcite.config import AzureConfig
    from sentcite.eval import evaluate_gt_set
    from sentcite.schema import GroundTruthItem

    cfg = AzureConfig.from_env()

    # Load a small synthetic GT slice from the cache bundle so we
    # have something to evaluate against without re-running 05a.
    synth_items_path = Path("../data/notebook_cache/synth_gt/items.jsonl")
    gt_items = []
    with synth_items_path.open() as f:
        for line in f:
            if not line.strip():
                continue
            d = json.loads(line)
            gt_items.append(GroundTruthItem(**d))

    gt_items = gt_items[:LIMIT]
    print(f"Running live eval on {len(gt_items)} GT items. "
          f"This will make Azure calls.")

    report = evaluate_gt_set(
        gt_items,
        cfg=cfg,
        strategies=("inline_prompted", "post_gen_alignment"),
        faithfulness=True,
        self_consistency=True,
        reviews=None,   # set REVIEWS_PATH above and load here if desired
    )
    print(report.to_markdown_table())
else:
    print("RESUME_FROM_CACHE=True — skipping live eval. "
          "Flip the knob at the top of the notebook to re-run live.")


## Part 7 — How this ties into Phase 1b

**What Phase 1a proves (this notebook):**

- The full pipeline — ingest → chunk → index → retrieve → generate
  → cite → evaluate — runs end-to-end and emits defensible metrics.
- The three-distinct-roles invariant (RAG generator ≠ synth-GT
  author ≠ judge) is enforced and produces non-degenerate
  faithfulness numbers.
- We can compare two citation strategies on the *same* questions,
  with *separate* F1 / faithfulness / stability signals — so a
  single metric never dominates the decision.
- F1 pessimism is quantified: the gap between F1 and faithfulness
  is the "parallel sentence" credit we're not giving ourselves.

**What Phase 1b needs:**

- **SME-authored GT** — questions and gold answers written by a tax
  SME, not synthesized by a model. Replaces `05a_synthetic_gt.ipynb`
  output for headline reporting.
- **SME faithfulness judgments** — the LLM judge calibrated against
  SME labels on a held-out slice.
- **Failure-mode review** — structured review of the bottom decile
  by faithfulness, stability, and reviewer confidence.
- **Inter-annotator agreement** — before we cite any SME number as a
  ground truth, we need ≥2 SMEs and Cohen's κ.

See `docs/status.md` (Phase 1b tasks are tagged `blocked` in the
planning board) and `docs/phase-1a/` for the methodology write-up.


## What's next

- **`05b_public_benchmark.ipynb`** — run the same sentence-attribution
  eval against a public benchmark (HAGRID-shape fixture), so our
  numbers are comparable to published work, not just our own synth-GT.
- **`demo_03_metrics_that_matter.ipynb`** — the 10-minute customer
  version of this notebook: Strategy A vs B, faithfulness, the
  F1-pessimism framing, and why we keep both metrics side-by-side.
